In [277]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils import resample

from sklearn.utils.class_weight import compute_class_weight

import re
import nltk
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB  
from hmmlearn.hmm import GaussianHMM ,CategoricalHMM, MultinomialHMM


from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,ConfusionMatrixDisplay
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.decomposition import TruncatedSVD



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model

import joblib
from scipy.sparse import hstack

import spacy
import pickle


In [2]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nlp = spacy.load("en_core_web_sm")


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\20100\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Reading Data

In [3]:
data = pd.read_json("News_Category_Dataset_v3.json", lines=True)

In [4]:
data.head()	

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


## EDA


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [6]:
data.duplicated().sum()

13

In [7]:
data.drop_duplicates(inplace=True)

In [8]:
data['category'].value_counts()

category
POLITICS          35601
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3571
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2100
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1443
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [9]:
other=set()
def mapping_to_10(cat):
    if cat not in ["POLITICS","WELLNESS","ENTERTAINMENT","TRAVEL","STYLE & BEAUTY","QUEER VOICES","FOOD & DRINK","BUSINESS","SPORTS"]:
        other.add(cat)
        return "OTHER"
    else:
        return cat

In [10]:
data['category_10']=data['category'].apply(mapping_to_10)
data

,link,headline,category,short_description,authors,date,category_10
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23,OTHER
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23,OTHER
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23,OTHER
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23,OTHER
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22,OTHER
...,...,...,...,...,...,...,...
209522,https://www.huffingtonpost.com/entry/rim-ceo-t...,RIM CEO Thorsten Heins' 'Significant' Plans Fo...,TECH,Verizon Wireless and AT&T are already promotin...,"Reuters, Reuters",2012-01-28,OTHER
209523,https://www.huffingtonpost.com/entry/maria-sha...,Maria Sharapova Stunned By Victoria Azarenka I...,SPORTS,"Afterward, Azarenka, more effusive with the pr...",,2012-01-28,SPORTS
209524,https://www.huffingtonpost.com/entry/super-bow...,"Giants Over Patriots, Jets Over Colts Among M...",SPORTS,"Leading up to Super Bowl XLVI, the most talked...",,2012-01-28,SPORTS
209525,https://www.huffingtonpost.com/entry/aldon-smi...,Aldon Smith Arrested: 49ers Linebacker Busted ...,SPORTS,CORRECTION: An earlier version of this story i...,,2012-01-28,SPORTS


In [11]:
print(len(other))
print(other)

33
{'ARTS & CULTURE', 'GOOD NEWS', 'GREEN', 'TASTE', 'IMPACT', 'EDUCATION', 'BLACK VOICES', 'DIVORCE', 'CRIME', 'SCIENCE', 'COMEDY', 'COLLEGE', 'PARENTS', 'THE WORLDPOST', 'HOME & LIVING', 'ENVIRONMENT', 'MEDIA', 'TECH', 'LATINO VOICES', 'HEALTHY LIVING', 'ARTS', 'RELIGION', 'WORLD NEWS', 'PARENTING', 'WOMEN', 'U.S. NEWS', 'STYLE', 'FIFTY', 'WORLDPOST', 'WEIRD NEWS', 'WEDDINGS', 'CULTURE & ARTS', 'MONEY'}


In [12]:
data['category_10'].value_counts()

category_10
OTHER             95142
POLITICS          35601
WELLNESS          17942
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9811
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
SPORTS             5077
Name: count, dtype: int64

## Split train test

In [13]:
train_df, test_df = train_test_split(data,test_size=0.2,random_state=42,stratify=data['category_10'])

### Resampling the "Other" category to balance the dataset

In [14]:
train_df[train_df['category'].isin(other)]['category'].value_counts()

category
PARENTING         7040
HEALTHY LIVING    5382
COMEDY            4351
BLACK VOICES      3658
HOME & LIVING     3453
PARENTS           3153
THE WORLDPOST     2922
WEDDINGS          2909
WOMEN             2862
CRIME             2845
IMPACT            2775
DIVORCE           2730
WORLD NEWS        2599
MEDIA             2349
WEIRD NEWS        2206
RELIGION          2081
GREEN             2060
WORLDPOST         2053
STYLE             1810
SCIENCE           1763
TECH              1701
TASTE             1694
MONEY             1398
ARTS              1234
ENVIRONMENT       1157
GOOD NEWS         1140
FIFTY             1107
U.S. NEWS         1093
ARTS & CULTURE    1083
COLLEGE            923
LATINO VOICES      895
CULTURE & ARTS     869
EDUCATION          818
Name: count, dtype: int64

In [15]:
818*33

26994

In [16]:
train_df['category_10'].value_counts()

category_10
OTHER             76113
POLITICS          28481
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
QUEER VOICES       5078
FOOD & DRINK       5072
BUSINESS           4794
SPORTS             4062
Name: count, dtype: int64

In [17]:
non_other =train_df[train_df['category_10']!="OTHER"]
non_other['category_10'].value_counts() 

category_10
POLITICS          28481
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
QUEER VOICES       5078
FOOD & DRINK       5072
BUSINESS           4794
SPORTS             4062
Name: count, dtype: int64

In [18]:
samples_number=train_df['category'].value_counts().min()
samples_number

818

In [19]:
balanced_data=[]
for sub in other:
    dfsub= train_df[train_df['category']==sub]
    if len(dfsub)>samples_number:
        dfsub_downsample=resample(dfsub,n_samples=samples_number,random_state=42)
    else:
        dfsun_downsample= dfsub
    balanced_data.append(dfsub_downsample)

In [20]:
train_df_balanced=pd.concat(balanced_data+[non_other])
train_df_balanced['category_10'].value_counts()

category_10
POLITICS          28481
OTHER             26994
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
QUEER VOICES       5078
FOOD & DRINK       5072
BUSINESS           4794
SPORTS             4062
Name: count, dtype: int64

In [21]:
print("Train distribution (10 classes):")
print(train_df_balanced['category_10'].value_counts())

print("\nTest distribution (10 classes):")
print(test_df['category_10'].value_counts())

Train distribution (10 classes):
category_10
POLITICS          28481
OTHER             26994
WELLNESS          14353
ENTERTAINMENT     13889
TRAVEL             7920
STYLE & BEAUTY     7849
QUEER VOICES       5078
FOOD & DRINK       5072
BUSINESS           4794
SPORTS             4062
Name: count, dtype: int64

Test distribution (10 classes):
category_10
OTHER             19029
POLITICS           7120
WELLNESS           3589
ENTERTAINMENT      3473
TRAVEL             1980
STYLE & BEAUTY     1962
QUEER VOICES       1269
FOOD & DRINK       1268
BUSINESS           1198
SPORTS             1015
Name: count, dtype: int64


In [22]:
joblib.dump({
    'X_train': train_df_balanced['headline'],
    'y_train': train_df_balanced['category_10'],
    'X_test': test_df['headline'],
    'y_test': test_df['category_10']
}, 'OTHERresample.pkl')

['OTHERresample.pkl']

## Use class wieghts to balance "train_df_balanced"

In [23]:
y_train = train_df_balanced['category_10']
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(zip(np.unique(y_train), class_weights))
class_weights_dict

{'BUSINESS': 2.4716729244889444,
 'ENTERTAINMENT': 0.8531355749154007,
 'FOOD & DRINK': 2.336198738170347,
 'OTHER': 0.4389568052159739,
 'POLITICS': 0.4160387626838945,
 'QUEER VOICES': 2.333438361559669,
 'SPORTS': 2.9170851797144266,
 'STYLE & BEAUTY': 1.5096445407058223,
 'TRAVEL': 1.4961111111111112,
 'WELLNESS': 0.8255556329687174}

## Preprocessing Text

In [24]:
lemmatizer=WordNetLemmatizer()
stop_words=set(stopwords.words('english'))

In [25]:
train_df_balanced.columns

Index(['link', 'headline', 'category', 'short_description', 'authors', 'date',
       'category_10'],
      dtype='object')

In [26]:
train_df_balanced.drop(columns=['short_description','category','date','authors','link'], inplace=True)
test_df.drop(columns=['short_description','category','date','authors','link'], inplace=True)

In [27]:
train_df_balanced.columns

Index(['headline', 'category_10'], dtype='object')

In [28]:
test_df.columns

Index(['headline', 'category_10'], dtype='object')

In [29]:
x_train = pd.DataFrame(train_df_balanced['headline'])
x_test = pd.DataFrame(test_df['headline'])
y_train = train_df_balanced['category_10']
y_test = test_df['category_10']

In [30]:
def preprocess(text):
    '''function to preprocess the text data
    1)    Converting the text to lowercase
    2)    Removing non-word characters
    3)   Removing extra spaces
    4)   Tokenization
    5)   Removing stop words
    6)   Lemmatization
    7)   Joining the tokens back to form the string    
    '''
    text = text.lower()

    text=re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) #remove URls
    text=re.sub(r'\@\w+|\#', '', text)                                    #remove special chars
    text=re.sub(r'[^a-zA-z\s]','',text)                                   #remove numbers

    tokens = word_tokenize(text)
    clean_tokens=[lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word)>2]
    
    return ' '.join(clean_tokens)

In [31]:
x_train["clean_text"] = x_train["headline"].apply(preprocess)
x_test["clean_text"] = x_test["headline"].apply(preprocess)

In [32]:
x_train

,headline,clean_text
59513,These Are The Scars Of People Who Have Narrowl...,scar people narrowly escaped death
94858,Meet The Last Pigeon Keeper in New York's East...,meet last pigeon keeper new york east village
74660,"Rebelling Against Borders, One Artist Is Paint...",rebelling border one artist painting immigrati...
34622,Neil Gaiman Will Dramatically Read Dr. Seuss I...,neil gaiman dramatically read seuss raise mill...
58588,Hulu's 'The Handmaid's Tale' Adds Joseph Fienn...,hulus handmaid tale add joseph fiennes banana
...,...,...
54614,Student Who Said Donald Trump Inspired His Hat...,student said donald trump inspired hate crime ...
138797,5 Secrets to Simplify Your Life,secret simplify life
25625,Steve Bannon: Chris Christie's Lack Of Loyalty...,steve bannon chris christie lack loyalty cost ...
27147,America's Founding Father Would Be Outraged By...,america founding father would outraged trump


In [34]:
joblib.dump({
    'X_train': x_train["clean_text"],
    'y_train': y_train,
    'X_test': x_test["clean_text"],
    'y_test': y_test
}, 'OTHERresample.pkl')

['OTHERresample.pkl']

## Feature Extaction

In [40]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

In [36]:
results = {}

In [35]:
num_classes = len(set(y_train))  

### BoW with countVectorizer

In [ ]:
BoW =CountVectorizer()

In [38]:
x_train_bow =BoW.fit_transform(x_train["clean_text"])
x_test_bow= BoW.transform(x_test["clean_text"])

In [39]:
print("BoW shape:", x_train_bow.shape)

BoW shape: (118492, 44186)


In [71]:
models_BOW={
    "SVC_BoW": SVC(class_weight=class_weights_dict),
    "MultinomialNB_BoW": MultinomialNB(),
    }

In [210]:
for model_name, model in models_BOW.items():
    model.fit(x_train_bow, y_train)
    y_pred = model.predict(x_test_bow)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_BoW is done
MultinomialNB_BoW is done


#### Models

In [ ]:
for model_name, model in models_BOW.items():
    model.fit(x_train_bow, y_train)

    y_pred_train = model.predict(x_train_bow)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_bow)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results[model_name] = accuracy

In [173]:
svc_noCW=SVC()
svc_noCW.fit(x_train_bow, y_train)
y_pred_train = svc_noCW.predict(x_train_bow)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_bow)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


Results for Training: accuracy=0.9010566114168045
Results for Testing: accuracy=0.6703815955898146
                precision    recall  f1-score   support

      BUSINESS       0.53      0.37      0.44      1198
 ENTERTAINMENT       0.59      0.72      0.65      3473
  FOOD & DRINK       0.62      0.67      0.64      1268
         OTHER       0.76      0.60      0.67     19029
      POLITICS       0.68      0.84      0.75      7120
  QUEER VOICES       0.80      0.62      0.70      1269
        SPORTS       0.66      0.59      0.62      1015
STYLE & BEAUTY       0.69      0.75      0.72      1962
        TRAVEL       0.71      0.69      0.70      1980
      WELLNESS       0.47      0.74      0.58      3589

      accuracy                           0.67     41903
     macro avg       0.65      0.66      0.65     41903
  weighted avg       0.69      0.67      0.67     41903

--------------------------------------------------


In [218]:
svc_noCW.fit(x_train_bow, y_train)
y_pred = svc_noCW.predict(x_test_bow)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()XBOW_confusion_matrix.png", dpi=300)
plt.close()


In [175]:
results['SVC_BoW']=0.6703815955898146

In [ ]:
results

{'SVC_BoW': 0.6345369066653939, 'MultinomialNB_BoW': 0.6365176717657447}

##### HMM + MLP

dimensionality reduction

In [43]:
svd_bow = TruncatedSVD(n_components=50, random_state=42)
x_train_svd_bow = svd_bow.fit_transform(x_train_bow)
x_test_svd_bow = svd_bow.transform(x_test_bow)

scaler = StandardScaler()
x_train_svd_bow = scaler.fit_transform(x_train_svd_bow)
x_test_svd_bow = scaler.transform(x_test_svd_bow)

In [ ]:
hmm_BoW=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_BoW=MLPClassifier(hidden_layer_sizes=(50,100,50),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)


In [58]:
MLP_BoW.fit(x_train_bow, y_train_enc)
y_pred_train = MLP_BoW.predict(x_train_bow)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_BoW Training: accuracy={accuracy_train}")

y_pred = MLP_BoW.predict(x_test_bow)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_BoW Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))

Results for MLP_BoW Training: accuracy=0.8645309387975559
Results for MLP_BoW Testing: accuracy=0.6384745722263322
              precision    recall  f1-score   support

           0       0.39      0.48      0.43      1198
           1       0.53      0.77      0.63      3473
           2       0.55      0.77      0.64      1268
           3       0.80      0.49      0.61     19029
           4       0.66      0.83      0.74      7120
           5       0.68      0.69      0.68      1269
           6       0.65      0.66      0.65      1015
           7       0.64      0.78      0.70      1962
           8       0.65      0.76      0.70      1980
           9       0.44      0.77      0.56      3589

    accuracy                           0.64     41903
   macro avg       0.60      0.70      0.64     41903
weighted avg       0.69      0.64      0.64     41903



In [223]:
MLP_BoW.fit(x_train_bow, y_train_enc)
y_pred = MLP_BoW.predict(x_test_bow)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXBOW_confusion_matrix.png", dpi=300)
plt.close()


In [51]:
hmm_BoW.fit(x_train_svd_bow)
y_pred_train = hmm_BoW.predict(x_train_svd_bow)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_BoW Training: accuracy={accuracy_train}")

y_pred = hmm_BoW.predict(x_test_svd_bow)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_BoW Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))

Results for hmm_BoW Training: accuracy=0.14838976470985382
Results for hmm_BoW Testing: accuracy=0.1820633367539317
              precision    recall  f1-score   support

           0       0.04      0.24      0.07      1198
           1       0.06      0.09      0.07      3473
           2       0.04      0.14      0.06      1268
           3       0.51      0.26      0.34     19029
           4       0.30      0.19      0.23      7120
           5       0.00      0.00      0.00      1269
           6       0.01      0.00      0.01      1015
           7       0.04      0.07      0.05      1962
           8       0.01      0.00      0.00      1980
           9       0.07      0.12      0.09      3589

    accuracy                           0.18     41903
   macro avg       0.11      0.11      0.09     41903
weighted avg       0.30      0.18      0.22     41903



In [230]:
hmm_BoW.fit(x_train_svd_bow)
y_pred = hmm_BoW.predict(x_test_svd_bow)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXBOW_confusion_matrix.png", dpi=300)
plt.close()

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


In [62]:
results['MLP_BoW']=0.6384745722263322
results['HMM_BoW']=0.1820633367539317
results

{'SVC_BoW': 0.6345369066653939,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317}

### BoW with tf-idfVectorizer

In [67]:
BoW_tfidf =TfidfVectorizer()

In [68]:
x_train_BoW_tfidf =BoW_tfidf.fit_transform(x_train["clean_text"])
x_test_BoW_tfidf= BoW_tfidf.transform(x_test["clean_text"])

In [69]:
print("BoW(tf-idf) shape:", x_train_BoW_tfidf.shape)

BoW(tf-idf) shape: (118492, 44186)


In [211]:
for model_name, model in models_BOW.items():
    model.fit(x_train_BoW_tfidf, y_train)
    y_pred = model.predict(x_test_BoW_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_tfidf_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")

        

SVC_BoW is done
MultinomialNB_BoW is done


#### Models

In [174]:
svc_noCW=SVC()
svc_noCW.fit(x_train_BoW_tfidf, y_train)
y_pred_train = svc_noCW.predict(x_train_BoW_tfidf)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_BoW_tfidf)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


Results for Training: accuracy=0.9367552239813658
Results for Testing: accuracy=0.6841992220127437
                precision    recall  f1-score   support

      BUSINESS       0.54      0.40      0.46      1198
 ENTERTAINMENT       0.60      0.74      0.66      3473
  FOOD & DRINK       0.62      0.68      0.65      1268
         OTHER       0.76      0.62      0.68     19029
      POLITICS       0.68      0.84      0.75      7120
  QUEER VOICES       0.80      0.64      0.71      1269
        SPORTS       0.66      0.64      0.65      1015
STYLE & BEAUTY       0.70      0.77      0.73      1962
        TRAVEL       0.73      0.70      0.72      1980
      WELLNESS       0.52      0.71      0.60      3589

      accuracy                           0.68     41903
     macro avg       0.66      0.67      0.66     41903
  weighted avg       0.70      0.68      0.68     41903

--------------------------------------------------


In [221]:
svc_noCW.fit(x_train_BoW_tfidf, y_train)
y_pred = svc_noCW.predict(x_test_BoW_tfidf)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()XBOW_tfidf_confusion_matrix.png", dpi=300)
plt.close()


In [176]:
results['(tf-idf) SVC_BoW']=0.6841992220127437

In [72]:
for model_name, model in models_BOW.items():
    model.fit(x_train_BoW_tfidf, y_train)

    y_pred_train = model.predict(x_train_BoW_tfidf)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_BoW_tfidf)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results['(tf-idf) '+model_name] = accuracy

Results for SVC_BoW Training: accuracy=0.9037909732302603
Results for SVC_BoW Testing: accuracy=0.6589981624227382
                precision    recall  f1-score   support

      BUSINESS       0.38      0.58      0.46      1198
 ENTERTAINMENT       0.56      0.77      0.65      3473
  FOOD & DRINK       0.57      0.78      0.66      1268
         OTHER       0.79      0.54      0.64     19029
      POLITICS       0.74      0.78      0.76      7120
  QUEER VOICES       0.73      0.70      0.71      1269
        SPORTS       0.62      0.71      0.66      1015
STYLE & BEAUTY       0.62      0.81      0.71      1962
        TRAVEL       0.65      0.78      0.71      1980
      WELLNESS       0.48      0.75      0.58      3589

      accuracy                           0.66     41903
     macro avg       0.61      0.72      0.65     41903
  weighted avg       0.69      0.66      0.66     41903

--------------------------------------------------
Results for MultinomialNB_BoW Training: accurac

In [ ]:
cm = confusion_matrix(y_test, y_pred)
y_pred = models_BOW["SVC_BoW"].predict(x_test_BoW_tfidf)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVC_BOW_tfidf_confusion_matrix.png", dpi=300)
plt.close()

##### HMM + MLP

dimensionality reduction

In [73]:
svd_bow_tfidf = TruncatedSVD(n_components=50, random_state=42)
x_train_svd_bow_tfidf = svd_bow_tfidf.fit_transform(x_train_BoW_tfidf)
x_test_svd_bow_tfidf = svd_bow_tfidf.transform(x_test_BoW_tfidf)

scaler = StandardScaler()
x_train_svd_bow_tfidf = scaler.fit_transform(x_train_svd_bow_tfidf)
x_test_svd_bow_tfidf = scaler.transform(x_test_svd_bow_tfidf)

In [ ]:
hmm_BoW=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_BoW=MLPClassifier(hidden_layer_sizes=(100,200,100,50),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [83]:
MLP_BoW.fit(x_train_svd_bow_tfidf, y_train_enc)
y_pred_train = MLP_BoW.predict(x_train_svd_bow_tfidf)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_BoW Training: accuracy={accuracy_train}")

y_pred = MLP_BoW.predict(x_test_svd_bow_tfidf)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_BoW Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))

Results for MLP_BoW Training: accuracy=0.6342537892853526
Results for MLP_BoW Testing: accuracy=0.5507720210963416
              precision    recall  f1-score   support

           0       0.35      0.17      0.23      1198
           1       0.38      0.53      0.45      3473
           2       0.51      0.53      0.52      1268
           3       0.64      0.52      0.58     19029
           4       0.59      0.76      0.67      7120
           5       0.50      0.43      0.46      1269
           6       0.39      0.22      0.28      1015
           7       0.58      0.62      0.60      1962
           8       0.54      0.50      0.52      1980
           9       0.41      0.56      0.47      3589

    accuracy                           0.55     41903
   macro avg       0.49      0.48      0.48     41903
weighted avg       0.56      0.55      0.55     41903



In [231]:
MLP_BoW.fit(x_train_svd_bow_tfidf, y_train_enc)
y_pred = MLP_BoW.predict(x_test_svd_bow_tfidf)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXBOW_tfidf_confusion_matrix.png", dpi=300)
plt.close()


In [76]:
hmm_BoW.fit(x_train_svd_bow_tfidf)
y_pred_train = hmm_BoW.predict(x_train_svd_bow_tfidf)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_BoW Training: accuracy={accuracy_train}")

y_pred = hmm_BoW.predict(x_test_svd_bow_tfidf)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_BoW Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))

Results for hmm_BoW Training: accuracy=0.12724065759713735
Results for hmm_BoW Testing: accuracy=0.12660191394410902
              precision    recall  f1-score   support

           0       0.03      0.15      0.05      1198
           1       0.11      0.25      0.16      3473
           2       0.02      0.09      0.03      1268
           3       0.52      0.13      0.21     19029
           4       0.23      0.16      0.19      7120
           5       0.01      0.01      0.01      1269
           6       0.01      0.05      0.02      1015
           7       0.04      0.07      0.05      1962
           8       0.00      0.00      0.00      1980
           9       0.09      0.09      0.09      3589

    accuracy                           0.13     41903
   macro avg       0.11      0.10      0.08     41903
weighted avg       0.30      0.13      0.15     41903



In [232]:
hmm_BoW.fit(x_train_svd_bow_tfidf)
y_pred = hmm_BoW.predict(x_test_svd_bow_tfidf)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXBOW_tfidf_confusion_matrix.png", dpi=300)
plt.close()


Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


In [84]:
results['(tf-idf) MLP_BoW']=0.5507720210963416
results['(tf-idf) HMM_BoW']=0.12660191394410902
results

{'SVC_BoW': 0.6345369066653939,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317,
 '(tf-idf) SVC_BoW': 0.6589981624227382,
 '(tf-idf) MultinomialNB_BoW': 0.613559888313486,
 '(tf-idf) MLP_BoW': 0.5507720210963416,
 '(tf-idf) HMM_BoW': 0.12660191394410902}

In [91]:
results["SVC_BoW"]=0.6345369066653939
results["MultinomialNB_BoW"]=0.6365176717657447
results["MLP_BoW"]=0.6384745722263322
results["HMM_BoW"]=0.1820633367539317

### POS with countVectorizer

In [85]:
models_POS={
    "SVC_POS": SVC(class_weight=class_weights_dict),
    "MultinomialNB_POS": MultinomialNB()
}

In [ ]:
def pos_features(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text.lower())

    
    tokens = [t for t in tokens if t not in stop_words]

    pos_tags = nltk.pos_tag(tokens)
    pos_tokens = [tag for _, tag in pos_tags]
    return pos_tokens


In [ ]:
pos=CountVectorizer(tokenizer=lambda x:pos_features(x))
x_train_pos =pos.fit_transform(x_train["clean_text"])
x_test_pos= pos.transform(x_test["clean_text"])

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [122]:
pos_counts = np.asarray(x_train_pos.sum(axis=0)).ravel()
pos_names = pos.get_feature_names_out()

pos_freq = pd.Series(pos_counts, index=pos_names).sort_values(ascending=False)
pos_freq

NN      448248
JJ      134807
VBG      29482
VBP      28028
NNS      26548
RB       24412
VBD      22220
VB       17742
VBN       9320
IN        8429
VBZ       8154
JJS       5280
CD        4970
MD        4653
JJR       2191
RBR       1267
DT        1220
RP         658
NNP        643
FW         571
CC         454
PRP        363
RBS        329
WRB        219
WP         121
WP$         60
PRP$        35
WDT         29
UH          21
EX           6
POS          6
''           5
TO           5
NNPS         2
``           1
dtype: int64

In [212]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos, y_train)
    y_pred = model.predict(x_test_pos)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_POS is done
MultinomialNB_POS is done


In [ ]:
svc_noCW=SVC()
svc_noCW.fit(x_train_pos, y_train)
y_pred_train = svc_noCW.predict(x_train_pos)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_pos)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


In [ ]:
svc_noCW.fit(x_train_pos, y_train)
y_pred = svc_noCW.predict(x_test_pos)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()POS_confusion_matrix.png", dpi=300)
plt.close()


In [172]:
results['SVC_POS']=0.34

In [123]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos, y_train)

    y_pred_train = model.predict(x_train_pos)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results[model_name] = accuracy

Results for SVC_POS Training: accuracy=0.1953127637308848
Results for SVC_POS Testing: accuracy=0.14679139918382933
                precision    recall  f1-score   support

      BUSINESS       0.03      0.03      0.03      1198
 ENTERTAINMENT       0.15      0.17      0.16      3473
  FOOD & DRINK       0.06      0.28      0.10      1268
         OTHER       0.54      0.04      0.08     19029
      POLITICS       0.25      0.23      0.24      7120
  QUEER VOICES       0.04      0.06      0.05      1269
        SPORTS       0.04      0.07      0.05      1015
STYLE & BEAUTY       0.10      0.34      0.15      1962
        TRAVEL       0.10      0.31      0.16      1980
      WELLNESS       0.19      0.34      0.25      3589

      accuracy                           0.15     41903
     macro avg       0.15      0.19      0.13     41903
  weighted avg       0.33      0.15      0.13     41903

--------------------------------------------------
Results for MultinomialNB_POS Training: accura

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.19      0.02      0.03      3473
  FOOD & DRINK       0.12      0.01      0.01      1268
         OTHER       0.49      0.32      0.39     19029
      POLITICS       0.18      0.67      0.29      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.20      0.00      0.01      1962
        TRAVEL       0.14      0.05      0.08      1980
      WELLNESS       0.13      0.07      0.09      3589

      accuracy                           0.27     41903
     macro avg       0.15      0.11      0.09     41903
  weighted avg       0.30      0.27      0.24     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##### HMM + MLP

In [126]:
svd_pos = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_pos = svd_pos.fit_transform(x_train_pos)
x_test_svd_pos = svd_pos.transform(x_test_pos)

scaler = StandardScaler()
x_train_svd_pos = scaler.fit_transform(x_train_svd_pos)
x_test_svd_pos = scaler.transform(x_test_svd_pos)

In [127]:
hmm_POS=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_POS=MLPClassifier(hidden_layer_sizes=(100,200,100,50),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [128]:
MLP_POS.fit(x_train_svd_pos, y_train_enc)
y_pred_train = MLP_POS.predict(x_train_svd_pos)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_POS Training: accuracy={accuracy_train}")

y_pred = MLP_POS.predict(x_test_svd_pos)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_POS Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['MLP_POS']=accuracy

Results for MLP_POS Training: accuracy=0.30591938696283294
Results for MLP_POS Testing: accuracy=0.3193088800324559
              precision    recall  f1-score   support

           0       0.09      0.00      0.00      1198
           1       0.18      0.03      0.05      3473
           2       0.17      0.01      0.01      1268
           3       0.49      0.39      0.44     19029
           4       0.23      0.64      0.34      7120
           5       0.15      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.14      0.04      0.06      1962
           8       0.17      0.09      0.12      1980
           9       0.21      0.30      0.25      3589

    accuracy                           0.32     41903
   macro avg       0.18      0.15      0.13     41903
weighted avg       0.32      0.32      0.29     41903



In [236]:
MLP_POS.fit(x_train_svd_pos, y_train_enc)
y_pred = MLP_POS.predict(x_test_svd_pos)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXPOS_confusion_matrix.png", dpi=300)
plt.close()


In [ ]:
hmm_POS.fit(x_train_svd_pos)
y_pred_train = hmm_POS.predict(x_test_svd_pos)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_POS Training: accuracy={accuracy_train}")

y_pred = hmm_POS.predict(x_test_svd_pos)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_POS Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['HMM_POS']=accuracy

Model is not converging.  Current: 3470180.5570838465 is not greater than 3470180.582744651. Delta is -0.025660804472863674


Results for hmm_POS Training: accuracy=0.06184383755865375
Results for hmm_POS Testing: accuracy=0.04453141779824833
              precision    recall  f1-score   support

           0       0.03      0.02      0.02      1198
           1       0.08      0.29      0.13      3473
           2       0.00      0.00      0.00      1268
           3       0.59      0.00      0.00     19029
           4       0.06      0.00      0.00      7120
           5       0.02      0.13      0.04      1269
           6       0.03      0.43      0.05      1015
           7       0.00      0.00      0.00      1962
           8       0.00      0.00      0.00      1980
           9       0.05      0.06      0.06      3589

    accuracy                           0.04     41903
   macro avg       0.09      0.09      0.03     41903
weighted avg       0.29      0.04      0.02     41903



In [238]:
hmm_POS.fit(x_train_svd_pos)
y_pred = hmm_POS.predict(x_test_svd_pos)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXPOS_confusion_matrix.png", dpi=300)
plt.close()


Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Model is not converging.  Current: 3470180.557084099 is not greater than 3470180.5827377443. Delta is -0.025653645396232605


In [ ]:
results

{'SVC_BoW': 0.6345369066653939,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317,
 '(tf-idf) SVC_BoW': 0.6589981624227382,
 '(tf-idf) MultinomialNB_BoW': 0.613559888313486,
 '(tf-idf) MLP_BoW': 0.5507720210963416,
 '(tf-idf) HMM_BoW': 0.12660191394410902,
 'SVC_POS': 0.14679139918382933,
 'MultinomialNB_POS': 0.27112617235042835,
 'MLP_POS': 0.3193088800324559,
 'HMM_POS': 0.04453141779824833,
 '(tf-idf) MLP_POS': 0.5273130802090542,
 '(tf-idf) hmm_POS': 0.1820633367539317,
 '(tf-idf )SVC_POS': 0.6589981624227382,
 '(tf-idf )MultinomialNB_POS': 0.613559888313486}

### POS with tf-idfVectorizer

In [132]:
pos_tfidf=TfidfVectorizer(tokenizer=lambda x: pos_features(x))
x_train_pos_tfidf =pos_tfidf.fit_transform(x_train["clean_text"])
x_test_pos_tfidf= pos_tfidf.transform(x_test["clean_text"])

d:\Anaconda\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [213]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_tfidf, y_train)
    y_pred = model.predict(x_test_pos_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_tfidf_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_POS is done
MultinomialNB_POS is done


In [171]:
svc_noCW=SVC()
svc_noCW.fit(x_train_pos_tfidf, y_train)
y_pred_train = svc_noCW.predict(x_train_pos_tfidf)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_pos_tfidf)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


Results for Training: accuracy=0.28022144954933664
Results for Testing: accuracy=0.31594396582583584


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.17      0.01      0.01      3473
  FOOD & DRINK       0.17      0.01      0.01      1268
         OTHER       0.48      0.42      0.45     19029
      POLITICS       0.21      0.69      0.32      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.55      0.00      0.01      1962
        TRAVEL       0.22      0.06      0.10      1980
      WELLNESS       0.22      0.06      0.09      3589

      accuracy                           0.32     41903
     macro avg       0.20      0.12      0.10     41903
  weighted avg       0.33      0.32      0.27     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [239]:
svc_noCW.fit(x_train_pos_tfidf, y_train)
y_pred = svc_noCW.predict(x_test_pos_tfidf)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()POS_tfidf_confusion_matrix.png", dpi=300)
plt.close()


In [133]:
for model_name, model in models_POS.items():
    model.fit(x_train_pos_tfidf, y_train)

    y_pred_train = model.predict(x_train_pos_tfidf)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_pos_tfidf)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results['(tf-idf )'+model_name] = accuracy

Results for SVC_POS Training: accuracy=0.1760625189886237
Results for SVC_POS Testing: accuracy=0.12939407679641077
                precision    recall  f1-score   support

      BUSINESS       0.03      0.07      0.04      1198
 ENTERTAINMENT       0.14      0.16      0.15      3473
  FOOD & DRINK       0.06      0.13      0.08      1268
         OTHER       0.54      0.04      0.08     19029
      POLITICS       0.25      0.13      0.17      7120
  QUEER VOICES       0.04      0.08      0.06      1269
        SPORTS       0.04      0.13      0.06      1015
STYLE & BEAUTY       0.09      0.23      0.13      1962
        TRAVEL       0.09      0.34      0.14      1980
      WELLNESS       0.17      0.40      0.24      3589

      accuracy                           0.13     41903
     macro avg       0.14      0.17      0.12     41903
  weighted avg       0.33      0.13      0.12     41903

--------------------------------------------------
Results for MultinomialNB_POS Training: accura

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##### HMM + MLP

In [134]:
svd_pos_tfidf = TruncatedSVD(n_components=35, random_state=42)
x_train_svd_pos_tfidf = svd_pos_tfidf.fit_transform(x_train_pos_tfidf)
x_test_svd_pos_tfidf = svd_pos_tfidf.transform(x_test_pos_tfidf)

scaler = StandardScaler()
x_train_svd_pos_tfidf = scaler.fit_transform(x_train_svd_pos_tfidf)
x_test_svd_pos_tfidf = scaler.transform(x_test_svd_pos_tfidf)

In [240]:
hmm_POS_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_POS_tfidf=MLPClassifier(hidden_layer_sizes=(70,140,70),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [241]:
MLP_POS_tfidf.fit(x_train_svd_pos_tfidf, y_train_enc)
y_pred_train = MLP_POS_tfidf.predict(x_train_svd_pos_tfidf)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_POS_tfidf Training: accuracy={accuracy_train}")

y_pred = MLP_POS_tfidf.predict(x_test_svd_pos_tfidf)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_POS_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) MLP_POS']=accuracy

cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXPOS_tfidf_confusion_matrix.png", dpi=300)
plt.close()


Results for MLP_POS_tfidf Training: accuracy=0.28553826418661177
Results for MLP_POS_tfidf Testing: accuracy=0.3105266925995752


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.18      0.02      0.04      3473
           2       0.00      0.00      0.00      1268
           3       0.49      0.38      0.43     19029
           4       0.22      0.64      0.33      7120
           5       0.00      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.15      0.01      0.02      1962
           8       0.22      0.06      0.10      1980
           9       0.18      0.26      0.22      3589

    accuracy                           0.31     41903
   macro avg       0.14      0.14      0.11     41903
weighted avg       0.31      0.31      0.28     41903



In [242]:
hmm_POS_tfidf.fit(x_train_svd_pos)
y_pred_train = hmm_POS_tfidf.predict(x_train_svd_pos)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_POS_tfidf Training: accuracy={accuracy_train}")

y_pred = hmm_POS_tfidf.predict(x_test_svd_pos)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_POS_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) hmm_POS']=accuracy

cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXPOS_tfidf_confusion_matrix.png", dpi=300)
plt.close()

Model is not converging.  Current: 3470180.5570838465 is not greater than 3470180.582744651. Delta is -0.025660804472863674


Results for hmm_POS_tfidf Training: accuracy=0.06184383755865375
Results for hmm_POS_tfidf Testing: accuracy=0.04453141779824833
              precision    recall  f1-score   support

           0       0.03      0.02      0.02      1198
           1       0.08      0.29      0.13      3473
           2       0.00      0.00      0.00      1268
           3       0.59      0.00      0.00     19029
           4       0.06      0.00      0.00      7120
           5       0.02      0.13      0.04      1269
           6       0.03      0.43      0.05      1015
           7       0.00      0.00      0.00      1962
           8       0.00      0.00      0.00      1980
           9       0.05      0.06      0.06      3589

    accuracy                           0.04     41903
   macro avg       0.09      0.09      0.03     41903
weighted avg       0.29      0.04      0.02     41903



In [140]:
joblib.dump(results,'OTHER resampled results')
results

{'SVC_BoW': 0.6345369066653939,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317,
 '(tf-idf) SVC_BoW': 0.6589981624227382,
 '(tf-idf) MultinomialNB_BoW': 0.613559888313486,
 '(tf-idf) MLP_BoW': 0.5507720210963416,
 '(tf-idf) HMM_BoW': 0.12660191394410902,
 'SVC_POS': 0.14679139918382933,
 'MultinomialNB_POS': 0.27112617235042835,
 'MLP_POS': 0.3193088800324559,
 'HMM_POS': 0.04453141779824833,
 '(tf-idf) MLP_POS': 0.30317638355249027,
 '(tf-idf) hmm_POS': 0.04453141779824833,
 '(tf-idf )SVC_POS': 0.12939407679641077,
 '(tf-idf )MultinomialNB_POS': 0.27103071379137533}

### NER with countVectorizer

In [97]:
models_NER={
    "SVC_NER": SVC(class_weight=class_weights_dict),    
    "MultinomialNB_NER": MultinomialNB()
}

In [141]:
def NER_features(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    doc = nlp(text)
    return " ".join(ent.label_ for ent in doc.ents)

In [142]:
xtrain_NER_words = x_train["clean_text"].apply(NER_features)
xtest_NER_words = x_test["clean_text"].apply(NER_features)

In [115]:
xtrain_NER_words #" ".join(ent.label_ for ent in doc.ents)

59513                  
94858           GPE GPE
74660          CARDINAL
34622      ORG CARDINAL
58588            PERSON
              ...      
54614            PERSON
138797                 
25625     PERSON PERSON
27147               GPE
204769                 
Name: clean_text, Length: 118492, dtype: object

In [ ]:
xtrain_NER_words #[ent.label_ for ent in doc.ents]

59513                   []
94858           [GPE, GPE]
74660           [CARDINAL]
34622      [ORG, CARDINAL]
58588             [PERSON]
                ...       
54614             [PERSON]
138797                  []
25625     [PERSON, PERSON]
27147                [GPE]
204769                  []
Name: clean_text, Length: 118492, dtype: object

In [143]:
ner_word=CountVectorizer()

x_train_NER_words = ner_word.fit_transform(xtrain_NER_words)
x_test_NER_words = ner_word.transform(xtest_NER_words)

In [162]:
svc_noCW=SVC()
svc_noCW.fit(x_train_NER_words, y_train)
y_pred_train = svc_noCW.predict(x_train_NER_words)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_NER_words)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


Results for Training: accuracy=0.3160044559970293
Results for Testing: accuracy=0.40999451113285446


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.26      0.03      0.06      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.51      0.64      0.57     19029
      POLITICS       0.28      0.67      0.39      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.14      0.01      0.01      1962
        TRAVEL       0.21      0.04      0.07      1980
      WELLNESS       0.10      0.00      0.01      3589

      accuracy                           0.41     41903
     macro avg       0.15      0.14      0.11     41903
  weighted avg       0.33      0.41      0.33     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
svc_noCW.fit(x_train_NER_words, y_train)
y_pred = svc_noCW.predict(x_train_NER_words)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()NER_confusion_matrix.png", dpi=300)
plt.close()


In [228]:
results["SVC_NER"]=0.40999451113285446

In [144]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_words, y_train)

    y_pred_train = model.predict(x_train_NER_words)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_words)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results[model_name] = accuracy

Results for SVC_NER Training: accuracy=0.25663335921412417
Results for SVC_NER Testing: accuracy=0.18132353292127054
                precision    recall  f1-score   support

      BUSINESS       0.06      0.04      0.05      1198
 ENTERTAINMENT       0.20      0.40      0.27      3473
  FOOD & DRINK       0.03      0.00      0.00      1268
         OTHER       0.56      0.00      0.01     19029
      POLITICS       0.35      0.36      0.35      7120
  QUEER VOICES       0.05      0.02      0.02      1269
        SPORTS       0.07      0.03      0.04      1015
STYLE & BEAUTY       0.10      0.09      0.09      1962
        TRAVEL       0.14      0.23      0.17      1980
      WELLNESS       0.14      0.80      0.24      3589

      accuracy                           0.18     41903
     macro avg       0.17      0.20      0.12     41903
  weighted avg       0.36      0.18      0.12     41903

--------------------------------------------------
Results for MultinomialNB_NER Training: accur

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.11      0.00      0.00      1198
 ENTERTAINMENT       0.29      0.09      0.14      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.49      0.15      0.23     19029
      POLITICS       0.16      0.75      0.27      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.00      0.00      0.00      1962
        TRAVEL       0.14      0.01      0.01      1980
      WELLNESS       0.07      0.05      0.06      3589

      accuracy                           0.21     41903
     macro avg       0.13      0.10      0.07     41903
  weighted avg       0.29      0.21      0.17     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [247]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_words, y_train)
    y_pred = model.predict(x_test_NER_words)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_NER is done
MultinomialNB_NER is done


##### HMM + MLP

In [146]:
svd_ner = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_ner = svd_ner.fit_transform(x_train_NER_words)
x_test_svd_ner = svd_ner.transform(x_test_NER_words)

scaler = StandardScaler()
x_train_svd_ner = scaler.fit_transform(x_train_svd_ner)
x_test_svd_ner = scaler.transform(x_test_svd_ner)

In [147]:
hmm_NER=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_NER=MLPClassifier(hidden_layer_sizes=(70,140,70),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [148]:
MLP_NER.fit(x_train_svd_ner, y_train_enc)
y_pred_train = MLP_NER.predict(x_train_svd_ner)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_NER Training: accuracy={accuracy_train}")

y_pred = MLP_NER.predict(x_test_svd_ner)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_NER Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['MLP_NER']=accuracy

Results for MLP_NER Training: accuracy=0.3149410930695743
Results for MLP_NER Testing: accuracy=0.4100899696919075
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.25      0.03      0.06      3473
           2       0.00      0.00      0.00      1268
           3       0.51      0.64      0.57     19029
           4       0.28      0.67      0.39      7120
           5       0.00      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.17      0.01      0.01      1962
           8       0.21      0.04      0.06      1980
           9       0.10      0.00      0.01      3589

    accuracy                           0.41     41903
   macro avg       0.15      0.14      0.11     41903
weighted avg       0.33      0.41      0.33     41903



d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [248]:
MLP_NER.fit(x_train_svd_ner, y_train_enc)
y_pred = MLP_NER.predict(x_test_svd_ner)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXNER_confusion_matrix.png", dpi=300)
plt.close()


In [150]:
hmm_NER.fit(x_train_svd_ner)
y_pred_train = hmm_NER.predict(x_train_svd_ner)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for HMM_NER Training: accuracy={accuracy_train}")

y_pred = hmm_NER.predict(x_test_svd_ner)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for HMM_NER Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['HMM_NER']=accuracy

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Model is not converging.  Current: 2918881.3676820197 is not greater than 2918881.392970406. Delta is -0.025288386270403862


Results for HMM_NER Training: accuracy=0.05797859771123789
Results for HMM_NER Testing: accuracy=0.04119036823139155
              precision    recall  f1-score   support

           0       0.03      0.81      0.06      1198
           1       0.10      0.13      0.11      3473
           2       0.00      0.00      0.00      1268
           3       0.62      0.00      0.00     19029
           4       0.15      0.00      0.00      7120
           5       0.00      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.05      0.15      0.08      1962
           8       0.02      0.00      0.00      1980
           9       0.01      0.00      0.00      3589

    accuracy                           0.04     41903
   macro avg       0.10      0.11      0.03     41903
weighted avg       0.32      0.04      0.02     41903



In [249]:
hmm_NER.fit(x_train_svd_ner)
y_pred = hmm_NER.predict(x_test_svd_ner)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXNER_confusion_matrix.png", dpi=300)
plt.close()


Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'
Model is not converging.  Current: 2918881.3676820197 is not greater than 2918881.392970406. Delta is -0.025288386270403862


In [151]:
# joblib.dump(results,'OTHER resampled results')
results

{'SVC_BoW': 0.6345369066653939,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317,
 '(tf-idf) SVC_BoW': 0.6589981624227382,
 '(tf-idf) MultinomialNB_BoW': 0.613559888313486,
 '(tf-idf) MLP_BoW': 0.5507720210963416,
 '(tf-idf) HMM_BoW': 0.12660191394410902,
 'SVC_POS': 0.14679139918382933,
 'MultinomialNB_POS': 0.27112617235042835,
 'MLP_POS': 0.3193088800324559,
 'HMM_POS': 0.04453141779824833,
 '(tf-idf) MLP_POS': 0.30317638355249027,
 '(tf-idf) hmm_POS': 0.04453141779824833,
 '(tf-idf )SVC_POS': 0.12939407679641077,
 '(tf-idf )MultinomialNB_POS': 0.27103071379137533,
 'SVC_NER': 0.18132353292127054,
 'MultinomialNB_NER': 0.20716893778488413,
 'MLP_NER': 0.4100899696919075,
 'HMM_NER': 0.04119036823139155}

### NER with tf-idfVectorizer

In [152]:
ner_tfidf=TfidfVectorizer()

x_train_NER_tfidf = ner_tfidf.fit_transform(xtrain_NER_words)
x_test_NER_tfidf = ner_tfidf.transform(xtest_NER_words)

In [215]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_tfidf, y_train)
    y_pred = model.predict(x_test_NER_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_tfidf_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_NER is done
MultinomialNB_NER is done


In [164]:
svc_noCW=SVC()
svc_noCW.fit(x_train_NER_tfidf, y_train)
y_pred_train = svc_noCW.predict(x_train_NER_tfidf)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_NER_tfidf)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)


Results for Training: accuracy=0.3149073355163218
Results for Testing: accuracy=0.40739326539865883


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.26      0.04      0.06      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.51      0.63      0.57     19029
      POLITICS       0.28      0.67      0.39      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.19      0.00      0.01      1962
        TRAVEL       0.21      0.04      0.07      1980
      WELLNESS       0.10      0.00      0.01      3589

      accuracy                           0.41     41903
     macro avg       0.15      0.14      0.11     41903
  weighted avg       0.33      0.41      0.33     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [255]:
svc_noCW.fit(x_train_NER_tfidf, y_train)
y_pred = svc_noCW.predict(x_test_NER_tfidf)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM()NER_tfidf_confusion_matrix.png", dpi=300)
plt.close()

In [227]:
results['(tfidf) SVC_NER']=0.40739326539865883

In [153]:
for model_name, model in models_NER.items():
    model.fit(x_train_NER_tfidf, y_train)

    y_pred_train = model.predict(x_train_NER_tfidf)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_NER_tfidf)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results["(tf-idf) "+model_name] = accuracy

Results for SVC_NER Training: accuracy=0.2561438746919623
Results for SVC_NER Testing: accuracy=0.18220652459251127
                precision    recall  f1-score   support

      BUSINESS       0.06      0.04      0.05      1198
 ENTERTAINMENT       0.20      0.41      0.27      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.55      0.00      0.01     19029
      POLITICS       0.35      0.36      0.36      7120
  QUEER VOICES       0.05      0.01      0.02      1269
        SPORTS       0.07      0.01      0.02      1015
STYLE & BEAUTY       0.10      0.09      0.10      1962
        TRAVEL       0.14      0.24      0.18      1980
      WELLNESS       0.14      0.80      0.24      3589

      accuracy                           0.18     41903
     macro avg       0.17      0.20      0.12     41903
  weighted avg       0.35      0.18      0.12     41903

--------------------------------------------------
Results for MultinomialNB_NER Training: accura

d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM+ MLP

In [154]:
svd_ner_tfidf = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_ner_tfidf = svd_ner_tfidf.fit_transform(x_train_NER_tfidf)
x_test_svd_ner_tfidf = svd_ner_tfidf.transform(x_test_NER_tfidf)

scaler = StandardScaler()
x_train_svd_ner_tfidf = scaler.fit_transform(x_train_svd_ner_tfidf)
x_test_svd_ner_tfidf = scaler.transform(x_test_svd_ner_tfidf)

In [155]:
hmm_NER_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_NER_tfidf=MLPClassifier(hidden_layer_sizes=(70,140,70),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [156]:
MLP_NER_tfidf.fit(x_train_svd_ner_tfidf, y_train_enc)
y_pred_train = MLP_NER_tfidf.predict(x_train_svd_ner_tfidf)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_NER_tfidf Training: accuracy={accuracy_train}")

y_pred = MLP_NER_tfidf.predict(x_test_svd_ner_tfidf)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_NER_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) MLP_NER']=accuracy

Results for MLP_NER_tfidf Training: accuracy=0.31414779056813963
Results for MLP_NER_tfidf Testing: accuracy=0.407775099634871
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.26      0.03      0.06      3473
           2       0.17      0.00      0.00      1268
           3       0.51      0.63      0.57     19029
           4       0.28      0.67      0.39      7120
           5       0.00      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.25      0.00      0.01      1962
           8       0.21      0.04      0.06      1980
           9       0.10      0.00      0.01      3589

    accuracy                           0.41     41903
   macro avg       0.18      0.14      0.11     41903
weighted avg       0.34      0.41      0.33     41903



d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [250]:
MLP_NER_tfidf.fit(x_train_svd_ner_tfidf, y_train_enc)
y_pred = MLP_NER_tfidf.predict(x_test_svd_ner_tfidf)
cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXNER_tfidf_confusion_matrix.png", dpi=300)
plt.close()


In [159]:
hmm_NER_tfidf.fit(x_train_svd_ner_tfidf)
y_pred_train = hmm_NER_tfidf.predict(x_train_svd_ner_tfidf)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_NER_tfidf Training: accuracy={accuracy_train}")

y_pred = hmm_NER_tfidf.predict(x_test_svd_ner_tfidf)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_NER_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) HMM_NER']=accuracy

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'
Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


Results for hmm_NER_tfidf Training: accuracy=0.19343078013705567
Results for hmm_NER_tfidf Testing: accuracy=0.38061713958427795
              precision    recall  f1-score   support

           0       0.02      0.04      0.02      1198
           1       0.50      0.00      0.00      3473
           2       0.02      0.02      0.02      1268
           3       0.45      0.83      0.59     19029
           4       0.15      0.00      0.00      7120
           5       0.00      0.00      0.00      1269
           6       0.03      0.05      0.03      1015
           7       0.03      0.02      0.02      1962
           8       0.06      0.01      0.01      1980
           9       0.00      0.00      0.00      3589

    accuracy                           0.38     41903
   macro avg       0.13      0.10      0.07     41903
weighted avg       0.28      0.38      0.27     41903



In [251]:
y_pred = hmm_NER_tfidf.predict(x_test_svd_ner_tfidf)
cm = confusion_matrix(y_test_enc, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXNER_tfidf_confusion_matrix.png", dpi=300)
plt.close()

### POS + NER with countVectorizer

In [166]:
models_POS_NER={
    "SVC_POS_NER": SVC(),    
    "MultinomialNB_POS_NER": MultinomialNB()
}

In [167]:
x_train_POS_NER = hstack([x_train_pos, x_train_NER_words])
x_test_POS_NER  = hstack([x_test_pos, x_test_NER_words])

In [216]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER, y_train)
    y_pred = model.predict(x_test_POS_NER)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
        

SVC_POS_NER is done
MultinomialNB_POS_NER is done


In [186]:
svc_noCW=SVC(class_weight=class_weights_dict)

svc_noCW.fit(x_train_POS_NER, y_train)

y_pred_train = svc_noCW.predict(x_train_POS_NER)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_POS_NER)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)

Results for Training: accuracy=0.2544306788643959
Results for Testing: accuracy=0.20158461208028064
                precision    recall  f1-score   support

      BUSINESS       0.05      0.08      0.06      1198
 ENTERTAINMENT       0.21      0.33      0.25      3473
  FOOD & DRINK       0.06      0.37      0.11      1268
         OTHER       0.57      0.09      0.15     19029
      POLITICS       0.38      0.31      0.34      7120
  QUEER VOICES       0.06      0.12      0.08      1269
        SPORTS       0.06      0.05      0.06      1015
STYLE & BEAUTY       0.13      0.25      0.17      1962
        TRAVEL       0.21      0.22      0.21      1980
      WELLNESS       0.19      0.49      0.28      3589

      accuracy                           0.20     41903
     macro avg       0.19      0.23      0.17     41903
  weighted avg       0.38      0.20      0.20     41903

--------------------------------------------------


In [254]:
svc_noCW=SVC(class_weight=class_weights_dict)

svc_noCW.fit(x_train_POS_NER, y_train)
y_pred = svc_noCW.predict(x_test_POS_NER)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM_POS_NER_confusion_matrix.png", dpi=300)
plt.close()

In [168]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER, y_train)

    y_pred_train = model.predict(x_train_POS_NER)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results[model_name] = accuracy

Results for SVC_POS_NER Training: accuracy=0.3315329304931979
Results for SVC_POS_NER Testing: accuracy=0.40491134286327946


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.28      0.03      0.06      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.52      0.59      0.55     19029
      POLITICS       0.29      0.60      0.39      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.30      0.03      0.05      1962
        TRAVEL       0.36      0.11      0.17      1980
      WELLNESS       0.24      0.32      0.27      3589

      accuracy                           0.40     41903
     macro avg       0.20      0.17      0.15     41903
  weighted avg       0.36      0.40      0.36     41903

--------------------------------------------------
Results for MultinomialNB_POS_NER Training: accuracy=0.30457752422104445
Results for MultinomialNB_POS_NER Testing: accuracy=0.3894709209364485


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.11      0.00      0.00      1198
 ENTERTAINMENT       0.26      0.16      0.20      3473
  FOOD & DRINK       0.11      0.00      0.01      1268
         OTHER       0.51      0.59      0.55     19029
      POLITICS       0.28      0.54      0.37      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.23      0.02      0.04      1962
        TRAVEL       0.22      0.09      0.13      1980
      WELLNESS       0.16      0.13      0.14      3589

      accuracy                           0.39     41903
     macro avg       0.19      0.15      0.14     41903
  weighted avg       0.34      0.39      0.35     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


#### HMM+ MLP

In [178]:
svd_POS_NER = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_POS_NER = svd_POS_NER.fit_transform(x_train_POS_NER)
x_test_svd_POS_NER = svd_POS_NER.transform(x_test_POS_NER)

scaler = StandardScaler()
x_train_svd_POS_NER = scaler.fit_transform(x_train_svd_POS_NER)
x_test_svd_POS_NER = scaler.transform(x_test_svd_POS_NER)

In [179]:
hmm_POS_NER=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_POS_NER=MLPClassifier(hidden_layer_sizes=(70,140,70),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [180]:
MLP_POS_NER.fit(x_train_svd_POS_NER, y_train_enc)
y_pred_train = MLP_POS_NER.predict(x_train_svd_POS_NER)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_POS_NER Training: accuracy={accuracy_train}")

y_pred = MLP_POS_NER.predict(x_test_svd_POS_NER)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_POS_NER Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['MLP_POS_NER']=accuracy

Results for MLP_POS_NER Training: accuracy=0.344976876076022
Results for MLP_POS_NER Testing: accuracy=0.359186693076868
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.24      0.09      0.13      3473
           2       0.18      0.01      0.02      1268
           3       0.53      0.43      0.48     19029
           4       0.28      0.63      0.39      7120
           5       0.50      0.00      0.01      1269
           6       0.00      0.00      0.00      1015
           7       0.20      0.07      0.10      1962
           8       0.30      0.13      0.19      1980
           9       0.21      0.44      0.28      3589

    accuracy                           0.36     41903
   macro avg       0.25      0.18      0.16     41903
weighted avg       0.37      0.36      0.33     41903



d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [257]:
y_pred = MLP_POS_NER.predict(x_test_svd_POS_NER)
cm = confusion_matrix(y_test_enc, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXPOS_NER_confusion_matrix.png", dpi=300)
plt.close()

In [182]:
hmm_POS_NER.fit(x_train_svd_POS_NER)
y_pred_train = hmm_POS_NER.predict(x_train_svd_POS_NER)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_POS_NER Training: accuracy={accuracy_train}")

y_pred = hmm_POS_NER.predict(x_test_svd_POS_NER)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_POS_NER Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['HMM_POS_NER']=accuracy

Even though the 'startprob_' attribute is set, it will be overwritten during initialization because 'init_params' contains 's'
Even though the 'transmat_' attribute is set, it will be overwritten during initialization because 'init_params' contains 't'
Even though the 'means_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'm'


Even though the 'covars_' attribute is set, it will be overwritten during initialization because 'init_params' contains 'c'


Results for hmm_POS_NER Training: accuracy=0.07429193532052797
Results for hmm_POS_NER Testing: accuracy=0.06806195260482543
              precision    recall  f1-score   support

           0       0.02      0.03      0.02      1198
           1       0.10      0.05      0.07      3473
           2       0.05      0.18      0.08      1268
           3       0.47      0.05      0.09     19029
           4       0.15      0.06      0.08      7120
           5       0.03      0.05      0.04      1269
           6       0.02      0.24      0.04      1015
           7       0.02      0.02      0.02      1962
           8       0.04      0.18      0.06      1980
           9       0.06      0.08      0.07      3589

    accuracy                           0.07     41903
   macro avg       0.10      0.09      0.06     41903
weighted avg       0.26      0.07      0.08     41903



In [256]:
y_pred = hmm_POS_NER.predict(x_test_svd_POS_NER)
cm = confusion_matrix(y_test_enc, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXPOS_NER_confusion_matrix.png", dpi=300)
plt.close()

### POS + NER with tf-idfVectorizer

In [187]:
x_train_POS_NER_tfidf = hstack([x_train_pos_tfidf, x_train_NER_tfidf])
x_test_POS_NER_tfidf  = hstack([x_test_pos_tfidf, x_test_NER_tfidf])

In [217]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER_tfidf, y_train)
    y_pred = model.predict(x_test_POS_NER_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le.classes_
    )

    fig, ax = plt.subplots(figsize=(20,10))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix")
    plt.savefig(f"{model_name}_tfidf_confusion_matrix.png", dpi=300)
    plt.close()

    print (f"{model_name} is done")
    

SVC_POS_NER is done
MultinomialNB_POS_NER is done


In [188]:
for model_name, model in models_POS_NER.items():
    model.fit(x_train_POS_NER_tfidf, y_train)

    y_pred_train = model.predict(x_train_POS_NER_tfidf)
    accuracy_train=accuracy_score(y_train, y_pred_train)
    print(f"Results for {model_name} Training: accuracy={accuracy_train}")
    
    y_pred = model.predict(x_test_POS_NER_tfidf)
    accuracy=accuracy_score(y_test, y_pred)
    print(f"Results for {model_name} Testing: accuracy={accuracy}")
    print(classification_report(y_test, y_pred))
    print('-'*50)
    results["(tf-idf) "+model_name] = accuracy

Results for SVC_POS_NER Training: accuracy=0.33809877460081694
Results for SVC_POS_NER Testing: accuracy=0.3946256831253132


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.00      0.00      0.00      1198
 ENTERTAINMENT       0.26      0.15      0.19      3473
  FOOD & DRINK       0.12      0.00      0.01      1268
         OTHER       0.52      0.55      0.54     19029
      POLITICS       0.29      0.61      0.40      7120
  QUEER VOICES       1.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.29      0.04      0.07      1962
        TRAVEL       0.27      0.10      0.14      1980
      WELLNESS       0.22      0.26      0.24      3589

      accuracy                           0.39     41903
     macro avg       0.30      0.17      0.16     41903
  weighted avg       0.39      0.39      0.36     41903

--------------------------------------------------
Results for MultinomialNB_POS_NER Training: accuracy=0.31215609492623975
Results for MultinomialNB_POS_NER Testing: accuracy=0.41954036703815956


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                precision    recall  f1-score   support

      BUSINESS       0.12      0.00      0.00      1198
 ENTERTAINMENT       0.29      0.05      0.08      3473
  FOOD & DRINK       0.00      0.00      0.00      1268
         OTHER       0.51      0.68      0.58     19029
      POLITICS       0.29      0.59      0.39      7120
  QUEER VOICES       0.00      0.00      0.00      1269
        SPORTS       0.00      0.00      0.00      1015
STYLE & BEAUTY       0.09      0.00      0.00      1962
        TRAVEL       0.34      0.04      0.08      1980
      WELLNESS       0.22      0.06      0.09      3589

      accuracy                           0.42     41903
     macro avg       0.19      0.14      0.12     41903
  weighted avg       0.35      0.42      0.35     41903

--------------------------------------------------


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
svc_noCW=SVC(class_weight=class_weights_dict)

svc_noCW.fit(x_train_POS_NER_tfidf, y_train)

y_pred_train = svc_noCW.predict(x_train_POS_NER_tfidf)
accuracy_train=accuracy_score(y_train, y_pred_train)
print(f"Results for Training: accuracy={accuracy_train}")

y_pred = svc_noCW.predict(x_test_POS_NER_tfidf)
accuracy=accuracy_score(y_test, y_pred)
print(f"Results for Testing: accuracy={accuracy}")
print(classification_report(y_test, y_pred))
print('-'*50)

Results for Training: accuracy=0.26813624548492726
Results for Testing: accuracy=0.19633439133236283
                precision    recall  f1-score   support

      BUSINESS       0.05      0.07      0.06      1198
 ENTERTAINMENT       0.21      0.33      0.26      3473
  FOOD & DRINK       0.07      0.22      0.10      1268
         OTHER       0.58      0.05      0.10     19029
      POLITICS       0.36      0.36      0.36      7120
  QUEER VOICES       0.06      0.14      0.08      1269
        SPORTS       0.06      0.07      0.07      1015
STYLE & BEAUTY       0.13      0.18      0.15      1962
        TRAVEL       0.17      0.23      0.20      1980
      WELLNESS       0.17      0.59      0.27      3589

      accuracy                           0.20     41903
     macro avg       0.19      0.22      0.16     41903
  weighted avg       0.38      0.20      0.17     41903

--------------------------------------------------


In [253]:
svc_noCW=SVC(class_weight=class_weights_dict)

svc_noCW.fit(x_train_POS_NER_tfidf, y_train)
y_pred = svc_noCW.predict(x_test_POS_NER_tfidf)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"SVM_POS_NER_tfidf_confusion_matrix.png", dpi=300)
plt.close()

#### HMM+ MLP

In [190]:
svd_POS_NER_tfidf = TruncatedSVD(n_components=18, random_state=42)
x_train_svd_POS_NER_tfidf = svd_POS_NER_tfidf.fit_transform(x_train_POS_NER_tfidf)
x_test_svd_POS_NER_tfidf = svd_POS_NER_tfidf.transform(x_test_POS_NER_tfidf)

scaler = StandardScaler()
x_train_svd_POS_NER_tfidf = scaler.fit_transform(x_train_svd_POS_NER_tfidf)
x_test_svd_POS_NER_tfidf = scaler.transform(x_test_svd_POS_NER_tfidf)

In [191]:
hmm_POS_NER_tfidf=GaussianHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
MLP_POS_NER_tfidf=MLPClassifier(hidden_layer_sizes=(70,140,70),activation='relu',solver='adam',max_iter=100,alpha=0.001,early_stopping=True,random_state=42)

In [192]:
MLP_POS_NER_tfidf.fit(x_train_svd_POS_NER, y_train_enc)
y_pred_train = MLP_POS_NER_tfidf.predict(x_train_svd_POS_NER)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for MLP_POS_NER_tfidf Training: accuracy={accuracy_train}")

y_pred = MLP_POS_NER_tfidf.predict(x_test_svd_POS_NER)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for MLP_POS_NER_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) MLP_POS_NER']=accuracy

Results for MLP_POS_NER_tfidf Training: accuracy=0.344976876076022
Results for MLP_POS_NER_tfidf Testing: accuracy=0.359186693076868
              precision    recall  f1-score   support

           0       0.00      0.00      0.00      1198
           1       0.24      0.09      0.13      3473
           2       0.18      0.01      0.02      1268
           3       0.53      0.43      0.48     19029
           4       0.28      0.63      0.39      7120
           5       0.50      0.00      0.01      1269
           6       0.00      0.00      0.00      1015
           7       0.20      0.07      0.10      1962
           8       0.30      0.13      0.19      1980
           9       0.21      0.44      0.28      3589

    accuracy                           0.36     41903
   macro avg       0.25      0.18      0.16     41903
weighted avg       0.37      0.36      0.33     41903



d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [258]:
y_pred = MLP_POS_NER_tfidf.predict(x_test_svd_POS_NER)
cm = confusion_matrix(y_test_enc, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"MLPXPOS_NER_tfidf_confusion_matrix.png", dpi=300)
plt.close()

In [193]:
hmm_POS_NER_tfidf.fit(x_train_svd_POS_NER)
y_pred_train = hmm_POS_NER_tfidf.predict(x_train_svd_POS_NER)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_POS_NER_tfidf Training: accuracy={accuracy_train}")

y_pred = hmm_POS_NER_tfidf.predict(x_test_svd_POS_NER)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_POS_NER_tfidf Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['(tf-idf) HMM_POS_NER']=accuracy

Results for hmm_POS_NER_tfidf Training: accuracy=0.07429193532052797
Results for hmm_POS_NER_tfidf Testing: accuracy=0.06806195260482543
              precision    recall  f1-score   support

           0       0.02      0.03      0.02      1198
           1       0.10      0.05      0.07      3473
           2       0.05      0.18      0.08      1268
           3       0.47      0.05      0.09     19029
           4       0.15      0.06      0.08      7120
           5       0.03      0.05      0.04      1269
           6       0.02      0.24      0.04      1015
           7       0.02      0.02      0.02      1962
           8       0.04      0.18      0.06      1980
           9       0.06      0.08      0.07      3589

    accuracy                           0.07     41903
   macro avg       0.10      0.09      0.06     41903
weighted avg       0.26      0.07      0.08     41903



In [259]:
y_pred = hmm_POS_NER_tfidf.predict(x_test_svd_POS_NER)
cm = confusion_matrix(y_test_enc, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"HMMXPOS_NER_tfidf_confusion_matrix.png", dpi=300)
plt.close()

# categorical HMM

In [ ]:
hmm_cat_POS=MultinomialHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)
hmm_cat_NER=MultinomialHMM(n_components=num_classes, algorithm='viterbi', n_iter=100, random_state=42)

MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows the standard definition for a Multinomial distribution (e.g. as in https://en.wikipedia.org/wiki/Multinomial_distribution). See these issues for details:
https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
MultinomialHMM has undergone major changes. The previous version was implementing a CategoricalHMM (a special case of MultinomialHMM). This new implementation follows

In [285]:
xtrain=x_train_pos.toarray()
xtest=x_test_pos.toarray()

hmm_cat_POS.fit(xtrain)
y_pred_train = hmm_cat_POS.predict(xtrain)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for hmm_cat_POS Training: accuracy={accuracy_train}")

y_pred = hmm_cat_POS.predict(xtest)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for hmm_cat_POS Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['cat_HMM_POS']=accuracy

cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"cat_HMMXPOS_confusion_matrix.png", dpi=300)
plt.close()

Results for hmm_cat_POS Training: accuracy=0.0820983695101779
Results for hmm_cat_POS Testing: accuracy=0.06522206047299715
              precision    recall  f1-score   support

           0       0.03      0.08      0.04      1198
           1       0.07      0.01      0.02      3473
           2       0.03      0.05      0.04      1268
           3       0.43      0.02      0.04     19029
           4       0.20      0.09      0.12      7120
           5       0.04      0.17      0.06      1269
           6       0.03      0.10      0.04      1015
           7       0.04      0.40      0.08      1962
           8       0.02      0.00      0.01      1980
           9       0.09      0.11      0.10      3589

    accuracy                           0.07     41903
   macro avg       0.10      0.10      0.06     41903
weighted avg       0.25      0.07      0.06     41903



In [288]:
xtrain=x_train_NER_words.toarray()
xtest=x_test_NER_words.toarray()

hmm_cat_NER.fit(xtrain)
y_pred_train = hmm_NER.predict(xtrain)
accuracy_train=accuracy_score(y_train_enc, y_pred_train)
print(f"Results for HMM_NER Training: accuracy={accuracy_train}")

y_pred = hmm_NER.predict(xtest)
accuracy=accuracy_score(y_test_enc, y_pred)
print(f"Results for HMM_NER Testing: accuracy={accuracy}")
print(classification_report(y_test_enc, y_pred))
results['cat_HMM_NER']=accuracy

cm = confusion_matrix(y_test_enc, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)

fig, ax = plt.subplots(figsize=(20,10))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Confusion Matrix")
plt.savefig(f"cat_HMMXNER_confusion_matrix.png", dpi=300)
plt.close()

Results for HMM_NER Training: accuracy=0.07494176822063937
Results for HMM_NER Testing: accuracy=0.053576116268524925


d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Anaconda\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.02      0.04      0.03      1198
           1       0.09      0.15      0.12      3473
           2       0.00      0.00      0.00      1268
           3       0.00      0.00      0.00     19029
           4       0.00      0.00      0.00      7120
           5       0.00      0.00      0.00      1269
           6       0.00      0.00      0.00      1015
           7       0.05      0.85      0.09      1962
           8       0.00      0.00      0.00      1980
           9       0.00      0.00      0.00      3589

    accuracy                           0.05     41903
   macro avg       0.02      0.10      0.02     41903
weighted avg       0.01      0.05      0.01     41903



# Save the results

In [276]:
results

{'SVC_BoW': 0.6703815955898146,
 'MultinomialNB_BoW': 0.6365176717657447,
 'MLP_BoW': 0.6384745722263322,
 'HMM_BoW': 0.1820633367539317,
 '(tf-idf) SVC_BoW': 0.6841992220127437,
 '(tf-idf) MultinomialNB_BoW': 0.613559888313486,
 '(tf-idf) MLP_BoW': 0.5507720210963416,
 '(tf-idf) HMM_BoW': 0.12660191394410902,
 'SVC_POS': 0.34,
 'MultinomialNB_POS': 0.27112617235042835,
 'MLP_POS': 0.3193088800324559,
 'HMM_POS': 0.04453141779824833,
 '(tf-idf) MLP_POS': 0.3105266925995752,
 '(tf-idf) hmm_POS': 0.04453141779824833,
 '(tf-idf )SVC_POS': 0.12939407679641077,
 '(tf-idf )MultinomialNB_POS': 0.27103071379137533,
 'SVC_NER': 0.40999451113285446,
 'MultinomialNB_NER': 0.20716893778488413,
 'MLP_NER': 0.4100899696919075,
 'HMM_NER': 0.04119036823139155,
 '(tf-idf) SVC_NER': 0.18220652459251127,
 '(tf-idf) MultinomialNB_NER': 0.2101997470348185,
 '(tf-idf) MLP_NER': 0.407775099634871,
 '(tf-idf) HMM_NER': 0.38061713958427795,
 'SVC_POS_NER': 0.40491134286327946,
 'MultinomialNB_POS_NER': 0.3894

In [262]:
with open("OTHER resampled results.txt", "w", encoding="utf-8") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")

In [261]:
joblib.dump(results,'OTHER resampled results')

['OTHER resampled results']